# Tuning

Find the optimal `n_clusters` and `n_segments` for your data.
Unlike tsam's built-in tuning, this evaluates each candidate across **all slices**,
so the result is optimal for your full multi-dimensional dataset.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30)
print(f"Shape: {dict(da.sizes)}")

## Pareto front

Search across a grid of `(n_clusters, n_segments)` combinations up to a maximum
number of timesteps. Each combination is evaluated across all slices.

In [ ]:
result = tsam_xarray.find_pareto_front(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
)
print(f"Tested {len(result.history)} combinations")
print(f"Best: n_clusters={result.n_clusters}, n_segments={result.n_segments}")
print(f"Overall RMSE: {result.rmse:.4f}")

## Summary matrix

The RMSE for each tested `(n_clusters, n_segments)` combination as an xarray Dataset.

In [ ]:
result.summary_matrix

## RMSE heatmap

In [ ]:
result.summary_matrix["rmse"].plotly.imshow(
    x="n_segments",
    y="n_clusters",
    title="RMSE by (n_clusters, n_segments)",
)

## Best result

The best `AggregationResult` is ready to use — evaluated across all slices.

In [ ]:
result.best_result.accuracy.rmse.to_dataframe("RMSE")

In [ ]:
result.best_result.cluster_representatives.sel(
    scenario="low",
    variable="solar",
).plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Best typical periods (solar, low scenario)",
)

## Find optimal for a specific data reduction

`find_optimal_combination` tests only the boundary combos for a target reduction — faster.

In [ ]:
result_opt = tsam_xarray.find_optimal_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    data_reduction=0.05,
    show_progress=False,
)
print(f"Best for 5% reduction: {result_opt.n_clusters}c x {result_opt.n_segments}s")
print(f"RMSE: {result_opt.rmse:.4f}")
result_opt.summary